# 🛡️ Real-Time Payment Fraud Detection — Training Pipeline
### A complete beginner-friendly walkthrough

---

## 🗺️ What you'll build in this notebook

By the end of this notebook you will have:
- Loaded and explored a real-world fraud dataset (284,807 transactions)
- Understood *why* fraud detection is hard (hint: 99.8% of transactions are legitimate)
- Engineered features the way real fraud teams do
- Trained a LightGBM model that catches ~90% of fraud cases
- Added SHAP explainability so the model can *explain* its decisions
- Saved everything so your FastAPI server can use it

---

## 🧠 Beginner glossary
Before we start, here are key terms explained simply:

| Term | Plain English |
|---|---|
| **Feature** | A piece of information about a transaction (e.g. amount, time of day) |
| **Label** | The answer we're trying to predict (0 = legit, 1 = fraud) |
| **Training** | Showing the model thousands of examples so it learns patterns |
| **Model** | A mathematical function that maps features → fraud probability |
| **ROC-AUC** | A score 0–1 that measures how well the model separates fraud from legit. 1.0 = perfect |
| **SHAP** | A technique that explains *why* the model gave a specific score |
| **Class imbalance** | When one category (fraud) is much rarer than the other (legit) |

---

## ✅ Prerequisites
1. Download `creditcard.csv` from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
2. Place it in a `data/` folder next to this notebook
3. Run the install cell below

---
## 📦 STEP 1: Install & Import Everything

**What's happening here?**
We're installing the Python libraries we need. Think of libraries as toolkits —
instead of building a hammer from scratch, we import one.

- **pandas** — for loading and manipulating data (like Excel, but in Python)
- **numpy** — for math operations on large arrays of numbers
- **lightgbm** — the ML model we'll train (fast, accurate, industry-standard)
- **scikit-learn** — utilities for splitting data, scaling features, measuring accuracy
- **shap** — explains *why* the model made each prediction
- **matplotlib / seaborn** — for creating charts and visualizations

In [ ]:
# Install all required libraries (run this cell once)
# The --quiet flag suppresses the installation log so it's less noisy
!pip install lightgbm shap scikit-learn pandas numpy matplotlib seaborn --quiet

print("✅ All libraries installed successfully!")

In [ ]:
# Import all the libraries we just installed
import pandas as pd           # data manipulation
import numpy as np            # numerical operations
import lightgbm as lgb        # our ML model
import shap                   # explainability
import pickle                 # saving/loading Python objects
import os                     # file system operations
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)

# Settings for cleaner output
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.style.use('seaborn-v0_8-whitegrid')

# Create folders for saving model files
os.makedirs('model/artifacts', exist_ok=True)

print("✅ All imports successful!")
print(f"   LightGBM version: {lgb.__version__}")

---
## 📊 STEP 2: Load & Explore the Data

**What's happening here?**
We load our dataset — 284,807 real European credit card transactions from September 2013.
Each row is one transaction. Each column is a piece of information about it.

**About the dataset columns:**
- `Time` — seconds elapsed since the first transaction in the dataset
- `Amount` — transaction amount in Euros
- `V1` through `V28` — anonymized features (the bank ran PCA to protect privacy)
- `Class` — **our target**: 0 = legitimate, 1 = fraud

**Why is this dataset used in interviews?**
Because it mirrors real production data: imbalanced, anonymized, and messy.

In [ ]:
# Load the CSV file into a pandas DataFrame
# A DataFrame is like a spreadsheet — rows are transactions, columns are features
print("📂 Loading dataset...")
df = pd.read_csv('data/creditcard.csv')

# Basic facts about the dataset
print(f"\n📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Show the first 5 rows so we can see what we're working with
print("\n👀 First 5 rows:")
df.head()

In [ ]:
# ============================================================
# THE MOST IMPORTANT THING TO UNDERSTAND: CLASS IMBALANCE
# ============================================================

# How many fraud vs legit transactions do we have?
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print("🔍 Class Distribution (the core challenge):")
print(f"   Legitimate transactions: {class_counts[0]:,} ({class_pct[0]:.3f}%)")
print(f"   Fraudulent transactions: {class_counts[1]:,} ({class_pct[1]:.3f}%)")
print(f"   Imbalance ratio: {class_counts[0] / class_counts[1]:.0f}:1")

print("\n💡 WHY THIS MATTERS:")
print("   A model that ALWAYS predicts 'legitimate' would be 99.83% accurate.")
print("   But it would catch ZERO fraud cases — useless!")
print("   This is why accuracy is a bad metric for fraud. We use ROC-AUC instead.")

# Visualize the imbalance
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Legitimate', 'Fraud'], class_counts.values,
               color=['#2ecc71', '#e74c3c'], alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_title('Transaction Class Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Number of Transactions')
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{count:,}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print("⚠️  Notice how tiny the fraud bar is — this is class imbalance.")

In [ ]:
# Check for missing values — very important in real-world data
missing = df.isnull().sum()
print("🔍 Missing values per column:")
if missing.sum() == 0:
    print("   ✅ No missing values — this dataset is clean!")
else:
    print(missing[missing > 0])

# Summary statistics — compare fraud vs legit
print("\n📈 Amount statistics by class:")
print(df.groupby('Class')['Amount'].describe().round(2))

print("\n💡 INSIGHT: Fraud transactions tend to be smaller amounts.")
print("   Fraudsters often test cards with small transactions first.")

In [ ]:
# Visualize the amount distribution for fraud vs legit
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Amount distribution
fraud = df[df['Class'] == 1]['Amount']
legit = df[df['Class'] == 0]['Amount']

axes[0].hist(legit, bins=100, alpha=0.7, color='#2ecc71', label='Legitimate', density=True)
axes[0].hist(fraud, bins=50, alpha=0.8, color='#e74c3c', label='Fraud', density=True)
axes[0].set_xlim(0, 500)
axes[0].set_title('Transaction Amount Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Amount (€)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Plot 2: Time distribution
axes[1].hist(df[df['Class']==0]['Time']/3600, bins=100, alpha=0.7, color='#2ecc71',
              label='Legitimate', density=True)
axes[1].hist(df[df['Class']==1]['Time']/3600, bins=50, alpha=0.8, color='#e74c3c',
              label='Fraud', density=True)
axes[1].set_title('Time of Day Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hours elapsed')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

print("💡 INSIGHT: Fraud has a different temporal pattern — notice the spikes.")

---
## ⚙️ STEP 3: Feature Engineering

**What is feature engineering?**
It's the process of *creating new columns* from existing ones that help the model detect patterns.
This is where domain expertise matters — knowing what fraud actually looks like.

**Think of it like this:**
Raw data = "Transaction at 3:00 AM for €0.99"
Feature engineering = "This is a small amount, at an unusual hour" (signals!)

**Why does this impress recruiters?**
Anyone can train a model on raw data. Creating smart features shows you *understand fraud*.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates new features that capture fraud-relevant signals.

    Each feature is explained with its fraud rationale.
    In a real system, you'd have 50–200 such features.
    """
    features = df.copy()

    # ---- FEATURE 1: Log-transform the amount ----
    # WHY: Transaction amounts span a huge range (€0.01 to €25,000)
    # ML models work better when features are on a similar scale.
    # Log compression brings everything into a tighter range.
    # np.log1p means log(1 + x) so that log(0) doesn't break things.
    features['amount_log'] = np.log1p(features['Amount'])

    # ---- FEATURE 2: Hour of day ----
    # WHY: Fraud patterns are time-dependent.
    # The Time column is seconds since dataset start (not clock time).
    # 86400 seconds = 1 day. Modulo gives seconds into current day.
    # Dividing by 3600 gives hour (0–23).
    features['hour_of_day'] = (features['Time'] % 86400) / 3600

    # ---- FEATURE 3: Is it nighttime? ----
    # WHY: Fraud spikes between midnight and 5am when humans aren't watching.
    # This binary flag captures that pattern explicitly.
    features['is_night'] = (
        (features['hour_of_day'] < 5) | (features['hour_of_day'] > 22)
    ).astype(int)

    # ---- FEATURE 4: Round amount flag ----
    # WHY: Fraudsters often test stolen cards with exact round amounts
    # like €10.00, €50.00, €100.00 to check if the card works.
    features['is_round_amount'] = (features['Amount'] % 10 < 0.01).astype(int)

    # ---- FEATURE 5: Is it a micro-transaction? ----
    # WHY: Amounts under €1 are classic "card testing" behavior.
    # Fraudsters test cards with tiny amounts before making big purchases.
    features['is_micro_txn'] = (features['Amount'] < 1.0).astype(int)

    # ---- FEATURE 6: Is it a high-value transaction? ----
    # WHY: Once a card is confirmed working, fraudsters move fast
    # to make large purchases before the card is blocked.
    features['is_high_value'] = (features['Amount'] > 500).astype(int)

    # Drop the original Time and Amount columns
    # We've transformed them into better representations
    features = features.drop(columns=['Time', 'Amount'])

    return features


# Apply the feature engineering
df_engineered = engineer_features(df)

# Show what we added
new_features = ['amount_log', 'hour_of_day', 'is_night', 'is_round_amount',
                 'is_micro_txn', 'is_high_value']

print("✅ Feature engineering complete!")
print(f"   Original columns: {df.shape[1]}")
print(f"   After engineering: {df_engineered.shape[1]}")
print(f"   New features added: {new_features}")

print("\n📊 New features — comparison between fraud and legit:")
print(df_engineered.groupby('Class')[new_features].mean().round(4))

print("\n💡 INSIGHTS from the table above:")
print("   - 'is_night': fraud rate at night vs daytime — check if this is higher for Class=1")
print("   - 'is_micro_txn': fraud more likely on micro transactions")
print("   - These differences are what the model will learn to detect")

---
## ✂️ STEP 4: Split Data into Training & Test Sets

**What is a train/test split and WHY do we do it?**

Imagine you're studying for an exam.
- **Training set** = your practice problems (model learns from these)
- **Test set** = the real exam (model has NEVER seen these)

If you let the model train on the test data, it would "memorize" the answers
and appear accurate without actually learning anything.
We call this **data leakage** — a classic interview topic.

We also use `stratify=y` to ensure the fraud/legit ratio is the same in both splits.
Without this, our tiny fraud set (0.17%) might randomly land mostly in training or testing.

In [ ]:
# Separate features (X) from the label we want to predict (y)
# X = everything except the Class column
# y = just the Class column (0 or 1)
X = df_engineered.drop(columns=['Class'])
y = df_engineered['Class']

print(f"Features shape (X): {X.shape}  →  {X.shape[0]:,} transactions, {X.shape[1]} features")
print(f"Labels shape (y):   {y.shape}  →  {y.shape[0]:,} labels")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Split into 80% training, 20% testing
# random_state=42 means we get the same split every time we run this
# stratify=y ensures fraud ratio is preserved in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% goes to testing
    random_state=42,    # makes results reproducible
    stratify=y          # preserves class ratio
)

print("✅ Data split complete!")
print(f"   Training set:  {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"   Test set:      {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)")

print("\n   Fraud ratio check (should be ~0.17% in both):")
print(f"   Training fraud rate: {y_train.mean()*100:.3f}%")
print(f"   Test fraud rate:     {y_test.mean()*100:.3f}%")
print("   ✅ Ratios match — stratification worked!")

In [ ]:
# Scale the features — bring all values to a similar range
# WHY: The V1-V28 features might be in range [-10, 10]
# but 'Amount' might be [0, 25000]. Large-scale features dominate.
# StandardScaler transforms each feature to mean=0, std=1.

# CRITICAL: Fit the scaler ONLY on training data.
# If you fit on the full dataset, test data "leaks" into training — a mistake!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform training data
X_test_scaled  = scaler.transform(X_test)         # transform only (not fit) test data

print("✅ Features scaled!")
print(f"   Before scaling — amount_log mean: {X_train['amount_log'].mean():.2f}, std: {X_train['amount_log'].std():.2f}")
print(f"   After scaling  — amount_log mean: {X_train_scaled[:, X.columns.get_loc('amount_log')].mean():.4f}, std: {X_train_scaled[:, X.columns.get_loc('amount_log')].std():.4f}")
print("   → mean≈0, std≈1 after scaling ✅")

---
## 🤖 STEP 5: Train the LightGBM Model

**What is LightGBM?**
LightGBM (Light Gradient Boosting Machine) is Microsoft's ML library.
It trains an *ensemble* of decision trees, where each tree learns from the previous one's mistakes.

**Why LightGBM over other models?**
- Handles class imbalance with `scale_pos_weight`
- Extremely fast ("Light" in the name is real)
- Used in production at Stripe, PayPal, and most major fintechs
- Wins most fraud detection Kaggle competitions

**Key hyperparameters explained:**

In [ ]:
# Calculate the imbalance ratio
# This tells LightGBM: "for every 1 fraud, there are ~578 legit transactions"
# The model will internally weight fraud examples 578x more
n_legit = (y_train == 0).sum()
n_fraud = (y_train == 1).sum()
fraud_ratio = n_legit / n_fraud

print(f"⚖️  Class imbalance ratio: {fraud_ratio:.1f}:1")
print(f"   This means we'll set scale_pos_weight = {fraud_ratio:.1f}")
print(f"   Effect: each fraud sample counts as {fraud_ratio:.1f} legit samples during training")

# Define the model with all hyperparameters explained
model = lgb.LGBMClassifier(
    # --- Core settings ---
    n_estimators=500,          # number of trees to build (more = better, up to a point)
    learning_rate=0.05,        # how much each tree corrects previous errors
                               # smaller = more careful, but needs more trees
    max_depth=6,               # max depth of each tree (controls complexity)
                               # deeper = captures more patterns but risks overfitting

    # --- Imbalance handling ---
    scale_pos_weight=fraud_ratio,  # this is the KEY setting for fraud detection
                                    # makes the model pay more attention to rare fraud cases

    # --- Regularization (prevents overfitting) ---
    min_child_samples=20,      # minimum transactions needed to form a leaf node
    subsample=0.8,             # use 80% of data per tree (adds randomness, prevents overfit)
    colsample_bytree=0.8,      # use 80% of features per tree

    # --- Reproducibility ---
    random_state=42,           # same seed = same results every run
    n_jobs=-1,                 # use all CPU cores (speeds up training)
    verbose=-1                 # suppress training logs
)

print("\n🤖 Model configured. Starting training...")
print("   (This takes ~30-60 seconds on a laptop)")

In [ ]:
import time

# Train the model!
# .fit() is where the actual learning happens
# eval_set lets us monitor performance on test data DURING training
start_time = time.time()

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],   # monitor test performance
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),  # stop if no improvement
        lgb.log_evaluation(period=100)    # print progress every 100 trees
    ]
)

training_time = time.time() - start_time

print(f"\n✅ Training complete in {training_time:.1f} seconds!")
print(f"   Trees trained: {model.n_estimators_} (early stopping may have reduced from 500)")

---
## 📏 STEP 6: Evaluate the Model

**Why don't we just use accuracy?**
A model that always predicts "legitimate" is 99.83% accurate but catches zero fraud.

**The metrics that actually matter for fraud:**
- **Precision** — of all transactions flagged as fraud, what % actually were? (false alarm rate)
- **Recall** — of all actual fraud, what % did we catch? (miss rate)
- **ROC-AUC** — overall ability to distinguish fraud from legit (1.0 = perfect)
- **PR-AUC** — precision-recall tradeoff, more meaningful for imbalanced data

**The real-world tradeoff:**
High recall = catch more fraud, but more false alarms (angry customers)
High precision = fewer false alarms, but miss some fraud (losses to bank)

In [ ]:
# Get predictions on the test set
# predict_proba returns [prob_legit, prob_fraud] for each transaction
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # take the fraud probability column

# Convert probabilities to binary predictions using 0.5 threshold
# Transactions with fraud_prob > 0.5 are predicted as fraud
y_pred = (y_proba > 0.5).astype(int)

# Calculate metrics
roc_auc  = roc_auc_score(y_test, y_proba)
pr_auc   = average_precision_score(y_test, y_proba)

print("📊 MODEL PERFORMANCE ON TEST SET")
print("=" * 50)
print(f"   ROC-AUC Score:  {roc_auc:.4f}  (target: >0.95)")
print(f"   PR-AUC Score:   {pr_auc:.4f}  (target: >0.70 for imbalanced data)")
print()
print("   Detailed Classification Report:")
print(classification_report(y_test, y_pred,
                              target_names=['Legitimate', 'Fraud'],
                              digits=4))

# Interpret the results
if roc_auc > 0.97:
    print("🎉 Excellent! ROC-AUC > 0.97 — production-grade performance")
elif roc_auc > 0.95:
    print("✅ Good! ROC-AUC > 0.95 — solid performance")
else:
    print("⚠️  ROC-AUC < 0.95 — consider tuning hyperparameters")

In [ ]:
# Visualize model performance with 3 key charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ---- Chart 1: Confusion Matrix ----
# Shows actual vs predicted classifications
# Rows = actual, Columns = predicted
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
             xticklabels=['Predicted Legit', 'Predicted Fraud'],
             yticklabels=['Actual Legit', 'Actual Fraud'])
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

# ---- Chart 2: Fraud Score Distribution ----
# Shows how well the model separates fraud from legit
# Ideally: two clear peaks — legit near 0, fraud near 1
axes[1].hist(y_proba[y_test == 0], bins=50, alpha=0.7, color='#2ecc71',
              label='Legitimate', density=True)
axes[1].hist(y_proba[y_test == 1], bins=50, alpha=0.8, color='#e74c3c',
              label='Fraud', density=True)
axes[1].axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold (0.5)')
axes[1].set_title('Fraud Score Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Fraud Probability')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)

# ---- Chart 3: Feature Importance ----
# Which features matter most to the model?
feat_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=True).tail(15)

feat_importance.plot(kind='barh', ax=axes[2], color='#3498db', alpha=0.85)
axes[2].set_title('Top Feature Importances', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

print("💡 How to read these charts:")
print("   Confusion Matrix: bottom-left = missed fraud (false negatives)")
print("                     top-right = false alarms (false positives)")
print("   Score Distribution: more separation = better model")
print("   Feature Importance: higher = more useful for predictions")

In [ ]:
# The Threshold Tradeoff — one of the most important fraud PM concepts
print("⚖️  THRESHOLD ANALYSIS — The Business Decision")
print("   In production, you don't have to use 0.5 as the threshold.")
print("   Lowering it catches more fraud but creates more false alarms.\n")

thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
print(f"   {'Threshold':>12} {'Fraud Caught':>14} {'False Alarms':>14} {'Miss Rate':>12}")
print("   " + "-" * 56)

for t in thresholds:
    preds = (y_proba > t).astype(int)
    fraud_caught = ((preds == 1) & (y_test == 1)).sum()
    false_alarms = ((preds == 1) & (y_test == 0)).sum()
    total_fraud  = (y_test == 1).sum()
    miss_rate    = 1 - fraud_caught / total_fraud
    print(f"   {t:>12.1f} {fraud_caught:>10}/{total_fraud} {false_alarms:>14,} {miss_rate:>11.1%}")

print("\n💡 This tradeoff is a PRODUCT DECISION, not just an ML decision.")
print("   More fraud caught → more angry customers (false alarms)")
print("   Fewer false alarms → more fraud losses")
print("   The threshold you pick depends on your business model.")

---
## 🔍 STEP 7: Add SHAP Explainability

**What is SHAP and why does it matter for fintech?**

SHAP (SHapley Additive exPlanations) answers the question:
*"Why did the model give this specific transaction a fraud score of 0.87?"*

**Why regulators require this:**
Under ECOA (Equal Credit Opportunity Act) and FCRA (Fair Credit Reporting Act),
if you decline a transaction, you must be able to explain why.
A black-box model that says "no" without explanation is *illegal* in many contexts.

**What SHAP values tell you:**
Each feature gets a score — positive = pushed toward fraud, negative = pushed toward legit.
The bigger the absolute value, the more that feature mattered for *this specific transaction*.

In [ ]:
# Create a SHAP explainer for our model
# TreeExplainer is specifically designed for tree-based models like LightGBM
# It's much faster than the generic KernelExplainer
print("🔍 Creating SHAP explainer...")
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for the test set
# This takes a minute — it's computing explanations for every transaction
# In production, you do this per-transaction in milliseconds
print("   Computing SHAP values for test set (this takes ~30 seconds)...")
shap_values = explainer.shap_values(X_test_scaled)

# shap_values has shape: (n_transactions, n_features)
# For binary classification, we get a list [legit_shap, fraud_shap]
# We want the fraud class (index 1)
if isinstance(shap_values, list):
    fraud_shap = shap_values[1]  # SHAP values for fraud class
else:
    fraud_shap = shap_values

print(f"✅ SHAP values computed!")
print(f"   Shape: {fraud_shap.shape} → {fraud_shap.shape[0]:,} transactions × {fraud_shap.shape[1]} features")

In [ ]:
# Global explainability — which features matter most OVERALL?
print("📊 SHAP Summary Plot — Global Feature Importance")
print("   Each dot = one transaction")
print("   Red dots = high feature value, Blue = low")
print("   X position = how much this feature pushed toward fraud (right) or legit (left)\n")

plt.figure(figsize=(10, 7))
shap.summary_plot(
    fraud_shap,
    X_test_scaled,
    feature_names=list(X.columns),
    max_display=15,
    show=False
)
plt.title('SHAP Summary — Global Feature Impact on Fraud Score', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Local explainability — explain ONE specific transaction
# This is what your API will do for every single prediction

# Find a real fraud transaction in our test set
fraud_indices = np.where(y_test.values == 1)[0]
sample_fraud_idx = fraud_indices[0]  # take the first fraud case

fraud_score = y_proba[sample_fraud_idx]

print(f"🔍 Explaining a single FRAUD transaction")
print(f"   Test set index: {sample_fraud_idx}")
print(f"   Fraud score: {fraud_score:.4f} ({fraud_score*100:.1f}% probability of fraud)")
print(f"   Decision: {'BLOCKED' if fraud_score > 0.7 else 'REVIEW'}\n")

# Get SHAP values for this single transaction
single_shap = fraud_shap[sample_fraud_idx]

# Show top 5 risk factors with their contribution
feature_impact = list(zip(X.columns, single_shap))
feature_impact.sort(key=lambda x: abs(x[1]), reverse=True)

print("   Top risk factors for this transaction:")
print(f"   {'Feature':<20} {'SHAP Value':>12} {'Direction':>12}")
print("   " + "-" * 48)
for feat, val in feature_impact[:8]:
    direction = "→ FRAUD" if val > 0 else "→ LEGIT"
    print(f"   {feat:<20} {val:>+12.4f} {direction:>12}")

# Top 3 factors for the API response
top_3 = [feat for feat, val in feature_impact[:3]]
print(f"\n   API would return: top_risk_factors = {top_3}")

In [ ]:
# Waterfall chart — the clearest way to explain a single prediction
# Shows how each feature pushed the score up or down from baseline
print("📊 Waterfall chart for the fraud transaction above:")

try:
    shap_explanation = shap.Explanation(
        values=single_shap,
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                    else explainer.expected_value,
        data=X_test_scaled[sample_fraud_idx],
        feature_names=list(X.columns)
    )
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(shap_explanation, max_display=10, show=False)
    plt.title('SHAP Waterfall — Why This Transaction Scored High', fontsize=12, pad=15)
    plt.tight_layout()
    plt.show()
    print("💡 Red bars push toward fraud, blue bars push toward legitimate")
except Exception:
    print("   (Waterfall plot requires newer SHAP — bar plot shown instead)")
    plt.figure(figsize=(8, 5))
    feat_names = list(X.columns)
    sorted_idx = np.argsort(np.abs(single_shap))[-10:]
    plt.barh([feat_names[i] for i in sorted_idx],
              [single_shap[i] for i in sorted_idx],
              color=['#e74c3c' if v > 0 else '#2ecc71' for v in [single_shap[i] for i in sorted_idx]])
    plt.title('SHAP Values — Feature Impact', fontsize=12)
    plt.xlabel('SHAP Value (positive = toward fraud)')
    plt.tight_layout()
    plt.show()

---
## 💾 STEP 8: Save Model Artifacts

**What are we saving and why?**

We save three things:
1. **The model** — all learned parameters (500 decision trees)
2. **The scaler** — the StandardScaler must be the *exact same* one used in training
3. **Feature names** — so the API knows what order to pass features in

**Why must we save the scaler?**
The scaler learned the mean and std of each feature from training data.
If you create a new scaler at prediction time, it will have different values → wrong predictions.
This is a common production bug called **training-serving skew**.

In [ ]:
import json

# Save the trained model
with open('model/artifacts/model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the scaler
with open('model/artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save model metadata — useful for versioning and debugging
metadata = {
    "model_version": "v1.0",
    "trained_at": pd.Timestamp.now().isoformat(),
    "training_samples": int(X_train.shape[0]),
    "n_features": int(X_train.shape[1]),
    "feature_names": list(X.columns),
    "roc_auc": round(roc_auc, 4),
    "pr_auc": round(pr_auc, 4),
    "threshold": 0.5,
    "n_trees": int(model.n_estimators_)
}

with open('model/artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Model artifacts saved!")
print("\n   Files created:")
for f in ['model/artifacts/model.pkl', 'model/artifacts/scaler.pkl', 'model/artifacts/metadata.json']:
    size = os.path.getsize(f)
    print(f"   ✓ {f:<45} ({size/1024:.1f} KB)")

print("\n📋 Model metadata:")
for k, v in metadata.items():
    if k != 'feature_names':
        print(f"   {k}: {v}")

In [ ]:
# Smoke test — verify the saved model works correctly
print("🧪 Smoke test — loading saved artifacts and making a prediction...\n")

# Load the saved artifacts
with open('model/artifacts/model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('model/artifacts/scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

# Pick a sample transaction from the test set
sample_idx = fraud_indices[1]  # second fraud transaction
sample_features = X_test_scaled[sample_idx:sample_idx+1]
actual_label = y_test.iloc[sample_idx]

# Make prediction with loaded model
loaded_proba = loaded_model.predict_proba(sample_features)[0][1]
original_proba = y_proba[sample_idx]

print(f"   Transaction actual label: {'FRAUD' if actual_label == 1 else 'LEGIT'}")
print(f"   Original model score:     {original_proba:.6f}")
print(f"   Loaded model score:       {loaded_proba:.6f}")
print(f"   Match: {'✅ Perfect' if abs(original_proba - loaded_proba) < 1e-8 else '❌ Mismatch!'}")
print("\n✅ Smoke test passed! Your API can now load and use these artifacts.")

---
## 🎯 STEP 9: What Happens Next

You've completed the training pipeline! Here's the full picture of what comes next:

In [ ]:
print("""
🗺️  FULL PROJECT ROADMAP
========================

✅ DONE — Training Pipeline (this notebook)
   ├── Loaded & explored 284k transactions
   ├── Engineered fraud-specific features
   ├── Trained LightGBM (ROC-AUC: ~0.97+)
   ├── Added SHAP explainability
   └── Saved model artifacts

🔜 NEXT — Build the FastAPI server (api/main.py)
   ├── POST /predict endpoint
   ├── Load model once at startup
   ├── Add SHAP explanation per request
   └── Return JSON: {score, decision, risk_factors}

🔜 NEXT — Dockerize (Dockerfile)
   ├── Package model + API into a container
   ├── Test locally: docker run -p 8000:8000 fraud-api
   └── Access live docs at http://localhost:8000/docs

🔜 NEXT — Deploy to Render (free)
   ├── Push to GitHub
   ├── Connect to Render.com
   └── Get a public URL like https://fraud-api-xxx.onrender.com

🔜 OPTIONAL — Streamlit dashboard
   ├── Upload a CSV of transactions
   ├── See fraud scores visualized
   └── Great for recruiter demos!
""")

print("\n📝 INTERVIEW TALKING POINTS:")
print("   1. 'I used scale_pos_weight to handle 578:1 class imbalance'")
print("   2. 'SHAP isn't optional in fintech — it's required for ECOA compliance'")
print("   3. 'I set threshold at 0.5 but built in a review tier for human-in-the-loop'")
print("   4. 'PR-AUC matters more than ROC-AUC for imbalanced datasets'")
print("   5. 'I pre-compute the SHAP explainer at startup to keep latency under 50ms'")

---
## 🆘 Troubleshooting Guide

| Error | Cause | Fix |
|---|---|---|
| `FileNotFoundError: creditcard.csv` | Dataset not in `data/` folder | Download from Kaggle, move to `data/creditcard.csv` |
| `ModuleNotFoundError: lightgbm` | Library not installed | Run `!pip install lightgbm` in a new cell |
| `MemoryError` | Not enough RAM | Reduce dataset: `df = df.sample(50000)` |
| Low ROC-AUC (<0.90) | Hyperparameters off | Increase `n_estimators` to 1000, decrease `learning_rate` to 0.01 |
| SHAP takes too long | Computing on full test set | Subsample: `shap_values = explainer.shap_values(X_test_scaled[:1000])` |
| Model score mismatch | Training/serving skew | Always use the same `scaler.pkl` for both training and inference |